In [1]:
# LLM endpoint and catalog location - all secrets come from .env
import os
import re
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv
import oracledb

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# opencode inference endpoint (OpenAI-compatible chat completions)
LLM_URL = os.getenv("LLM_URL")
LLM_KEY = os.getenv("SQL_AGENT")     # API key stored under the name SQL_AGENT
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.8-flash")

# fail fast if the LLM config is missing
missing = [n for n, v in {"LLM_URL": LLM_URL, "SQL_AGENT": LLM_KEY}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
print(f"model: {LLM_MODEL} | catalog: {CATALOG_PATH.name}")

model: qwen3.8-flash | catalog: catalog.yaml


In [2]:
import time   # used by the retry backoff inside llm()

# Single model-agnostic wrapper: every agent calls this, so swapping
# providers later only means changing this one function.
# Credentials are read from .env at call time; 3 attempts with growing
# backoff because LLM endpoints are sometimes slow.
def llm(prompt, system="You are an expert SQL writer.", session="ses_queryllm",
        temperature=0.2, enable_thinking=False):
    payload = {
        "model": os.getenv("LLM_MODEL", "qwen3.8-flash"),
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        # qwen3.8-flash THINKS by default: ~12k hidden reasoning tokens were
        # 90% of the latency (55s -> 4-6s with thinking off). SQL generation
        # does not need it; the validator repair loop is the safety net.
        # EXCEPTION: the intent agent keeps thinking ON (short chain, better
        # routing accuracy) - it passes enable_thinking=True explicitly.
        "enable_thinking": enable_thinking,
    }
    headers = {
        "Authorization": f"Bearer {os.getenv('SQL_AGENT')}",
        "x-opencode-session": session,
        "Content-Type": "application/json",
    }
    last_error = None
    for attempt in range(3):
        try:
            response = requests.post(
                os.getenv("LLM_URL"), headers=headers, json=payload, timeout=120
            )
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"].strip()
        except (requests.Timeout, requests.ConnectionError) as exc:
            last_error = exc
            print(f"  llm call failed (attempt {attempt + 1}/3), retrying...")
            time.sleep(5 * (attempt + 1))
    raise last_error

In [3]:
# Load the catalog AND the PDP rules file (both live under catalog/)
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))
RULES_PATH = PROJECT_ROOT / "catalog" / "rules.yaml"
rules = yaml.safe_load(RULES_PATH.read_text(encoding="utf-8"))

print(f"rules: v{rules['rules_version']}")
for name, ws in catalog["workspaces"].items():
    print(f"{name}: {len(ws['tables'])} tables, {len(ws.get('examples', []))} examples")
    print(f"   {ws['description']}")

rules: v1.2
CO: 11 tables, 0 examples
   Customer orders, order items, shipments, stores, inventory, products and product reviews
HR: 8 tables, 0 examples
   Employees, departments, jobs, locations, regions and job history
SH: 12 tables, 0 examples
   Sales facts with product, customer, time, channel and promotion dimensions, plus profit and monthly-sales aggregations


In [4]:
# Render the rules file to prompt text once. Every agent below reads the
# same compliance text - no legal prose is hardcoded in the code itself.
def render_rules(rules):
    pdp = rules["pdp"]
    hints = pdp["classification_hints"]
    lines = ["PDP rules (columns are classified by name patterns):"]
    for cls in ("specific", "direct", "indirect"):
        pats = ", ".join(hints.get(cls, []))
        lines.append(f"- {cls.upper()} = column names matching: {pats}")
        lines.append(f"    handling -> {pdp['handling'][cls]}")
    lines.append("- everything else is NON_PERSONAL: no restriction")
    lines.append("Query rules:")
    lines.extend(f"- {rule}" for rule in rules["query_rules"])
    return "\n".join(lines)

rules_text = render_rules(rules)
print(rules_text)

PDP rules (columns are classified by name patterns):
- SPECIFIC = column names matching: salary, gaji, income, penghasilan, commission, diagnosa, nik, ktp, npwp, bpjs, rekening
    handling -> aggregate only (SUM/AVG/COUNT/MAX/MIN) - never select raw values, and suppress small cohorts using HAVING COUNT(*) >= 5
- DIRECT = column names matching: full_name, first_name, last_name, nama, email, phone, telp, no_hp, street_address, physical_address, delivery_address, alamat
    handling -> anonymize every direct column in SELECT using a collision-free Oracle hash mask - never return raw values or plain prefix-only masks like 'S***'. Use this exact Oracle SQL pattern (aliasing back to the column name): For emails: CASE WHEN col IS NOT NULL THEN SUBSTR(col, 1, 1) || '***_' || SUBSTR(RAWTOHEX(STANDARD_HASH(LOWER(TRIM(col)), 'SHA256')), 1, 8) || '@***.com' END AS col. For other direct columns: CASE WHEN col IS NOT NULL THEN SUBSTR(TO_CHAR(col), 1, 1) || '***_' || SUBSTR(RAWTOHEX(STANDARD_HASH(LO

In [5]:
# Turn one or more workspaces into a compact text schema for the prompt.\n# Object names are always schema-qualified (see the prefix note below).
# Columns are tagged with their PDP class so the model sees exactly which
# columns the rules apply to; the handling text comes from rules_text.
# - single workspace: plain table names (the session schema covers them)
# - several workspaces: SCHEMA.TABLE qualified names, plus the verified
#   FK joins that connect them
def pdp_class(column_name, rules):
    # word-boundary match on the lowercased column name
    hints = rules["pdp"]["classification_hints"]
    name = column_name.lower()
    for cls in ("specific", "direct", "indirect"):
        for pattern in hints.get(cls, []):
            if re.search(rf"(?<![a-z0-9]){re.escape(pattern)}(?![a-z0-9])", name):
                return cls.upper()
    return None

def schema_block(workspace_names, catalog, rules):
    names = [workspace_names] if isinstance(workspace_names, str) else list(workspace_names)
    lines = []
    for ws_name in names:
        ws = catalog["workspaces"][ws_name]
        # always qualify: SCHEMA.TABLE makes the generated SQL portable to any
        # SQL client (the analyst copies it out of the dashboard)
        prefix = f"{ws_name}."
        for table_name, table in ws["tables"].items():
            lines.append(f"{prefix}{table_name} ({table['object_type']}): {table['description'] or '-'}")
            for c in table["columns"]:
                cls = pdp_class(c["name"], rules)
                tag = f" [{cls}]" if cls else ""
                part = f"{c['name']} {c['type']}{tag}" + ("" if c["description"] == "" else f" -- {c['description']}")
                lines.append(f"    {part}")
            if table["primary_key"]:
                lines.append(f"    primary key: {', '.join(table['primary_key'])}")
            for fk in table["foreign_keys"]:
                if not fk["reference_resolved"]:
                    continue
                if len(names) == 1 or fk["references_schema"] == ws_name:
                    ref_prefix = "" if fk["references_schema"] == ws_name else fk["references_schema"] + "."
                    lines.append(
                        f"    FK {','.join(fk['columns'])} -> "
                        f"{ref_prefix}{fk['references_table']}.({','.join(fk['references_columns'])})"
                    )
    if len(names) > 1:
        for ws_name in names:
            for tname, table in catalog["workspaces"][ws_name]["tables"].items():
                for fk in table["foreign_keys"]:
                    if fk["reference_resolved"] and fk["references_schema"] != ws_name and fk["references_schema"] in names:
                        lines.append(
                            f"verified cross-workspace join: "
                            f"{ws_name}.{tname}.({','.join(fk['columns'])}) = "
                            f"{fk['references_schema']}.{fk['references_table']}.({','.join(fk['references_columns'])})"
                        )
        # candidate connections recorded in the catalog (never invented by the
        # model). 'unresolved' means the two schemas use different key names,
        # so the model must compare via descriptive columns instead of joining.
        for conn in catalog.get("connections", []):
            sides = {t.split(".", 1)[0] for t in conn["tables"]}
            if len(sides) == 2 and sides <= set(names):
                key = conn.get("join_key")
                if key and key != "unresolved":
                    lines.append(
                        f"candidate connection: {conn['tables'][0]} = {conn['tables'][1]} "
                        f"on {key} (candidate - the join values must match on both sides)"
                    )
                else:
                    lines.append(
                        f"candidate connection: {conn['tables'][0]} <-> {conn['tables'][1]} "
                        "has NO reliable join key - do not invent one; compare via descriptive columns only"
                    )
    return "\n".join(lines)

# quick look at the size of each block
for name in catalog["workspaces"]:
    print(f"{name}: {len(schema_block(name, catalog, rules))} chars")


def preselect_tables(question, workspace_name, catalog, max_tables=6):
    # the helpers live INSIDE the function so this cell exports verbatim to
    # src/writer.py (no module-level constants to carry along)
    def _tokens(text):
        return set(re.findall(r"[a-z]+", (text or "").lower()))

    def _hit(token, word):
        # plural-tolerant keyword match: store ~ stores, order ~ orders,
        # and prefix matches from 4+ chars (product ~ products)
        t, w = token.rstrip("s"), word.rstrip("s")
        return t == w or (len(t) >= 4 and (w.startswith(t) or t.startswith(w)))
    # Deterministic table preselection (NO LLM call): keep the tables the
    # question plausibly needs plus their FK-linked neighbors, so the writer
    # prompt shrinks and generation gets faster. Skipped entirely when the
    # whole workspace already fits (CO/HR keep all their tables).
    # Returns (pruned_tables, ranked_names) - (None, None) when nothing to prune.
    tables = catalog["workspaces"][workspace_name]["tables"]
    if len(tables) <= max_tables:
        return None, None
    tokens = _tokens(question)
    scores = {}
    for tname, table in tables.items():
        score = 0
        for t in tokens:
            if any(_hit(t, w) for w in _tokens(tname)):
                score += 3
            elif any(_hit(t, w) for w in _tokens(" ".join(c["name"] for c in table["columns"]))):
                score += 2
            elif any(_hit(t, w) for w in _tokens(table["description"])):
                score += 1
        scores[tname] = score
    ranked = sorted(scores, key=scores.get, reverse=True)
    keep = set(ranked[:max_tables])
    # FK neighbors of kept tables count as reachable context when they scored > 0
    for tname in list(keep):
        for fk in tables[tname]["foreign_keys"]:
            ref = fk["references_table"]
            if fk["reference_resolved"] and ref in tables and scores.get(ref, 0) > 0:
                keep.add(ref)
    pruned = {t: tables[t] for t in sorted(keep)}
    return pruned, ranked


CO: 6438 chars
HR: 6280 chars
SH: 7718 chars


In [6]:
# The SQL writer agent: question + one or more workspace schemas + the policy
# -> one Oracle SQL query. Rules force SELECT-only, Oracle dialect, and no
# invented tables or columns. Cross-schema behaviour follows the catalog
# policy: disallowed (one workspace) or verified_fk_only (multi-workspace).
def sql_rules(policy):
    if policy["cross_schema_joins"] in ("verified_fk_only", "catalog_connections_only"):
        # catalog_connections_only: joins come from candidate connections
        # recorded in catalog.yaml (never invented by the model)
        cross = ("Cross-schema joins are allowed, but ONLY using the joins listed "
                 "in the schema (verified cross-workspace joins or candidate "
                 "connections with their recorded key), and table names must be "
                 "qualified as SCHEMA.TABLE.")
    else:
        cross = "Cross-schema joins are NOT allowed: answer within ONE workspace."
    return f"""Rules:
- Write ONE query in Oracle SQL (12c+ syntax, FETCH FIRST for limits).
- SELECT or WITH only. Never INSERT/UPDATE/DELETE/DROP/ALTER/TRUNCATE.
- Use only tables and columns listed in the schema. Never invent names.
- Always qualify table names as SCHEMA.TABLE (e.g. CO.ORDERS) so the SQL runs unchanged in any SQL client.
- {cross}
- Return ONLY the SQL query, no explanations, no markdown fences."""

def sql_writer(question, workspace_names, catalog, rules):
    names = [workspace_names] if isinstance(workspace_names, str) else list(workspace_names)
    # respect the policy: multiple workspaces only when the catalog allows it
    if len(names) > 1 and catalog["policy"]["cross_schema_joins"] == "disallowed":
        raise ValueError("policy disallows cross-schema joins: pass a single workspace")
    # deterministic preselection: shrink the prompt to the tables the
    # question plausibly needs (single workspace only; multi-workspace is
    # policy-gated and dormant). When everything fits, the view is unchanged.
    used_catalog = catalog
    if len(names) == 1:
        pruned, _ = preselect_tables(question, names[0], catalog)
        if pruned is not None:
            used_catalog = {
                **catalog,
                "workspaces": {
                    names[0]: {**catalog["workspaces"][names[0]], "tables": pruned}
                },
            }
    schema = schema_block(names, used_catalog, rules)
    examples = "\n\n".join(
        f"Example: {ex['question']}\n{ex['sql']}"
        for ex in catalog["workspaces"][names[0]].get("examples", [])
    )
    label = ", ".join(names) if len(names) > 1 else names[0]
    rules_text = render_rules(rules)
    prompt = (
        f"{rules_text}\n\n"
        f"{sql_rules(catalog['policy'])}\n\n"
        f"Schema ({label}):\n{schema}\n\n"
        + (f"Few-shot examples:\n{examples}\n\n" if examples else "")
        + f"Question: {question}\n"
        "SQL:"
    )
    answer = llm(prompt, system="You are an expert SQL writer.", session="ses_queryllm_sql")
    # strip markdown fences if the model added them anyway
    answer = re.sub(r"^```(?:sql)?\s*|\s*```$", "", answer.strip(), flags=re.IGNORECASE)
    return answer.strip()


In [7]:
# Try the agent on one question per workspace and print the generated SQL.
# Multi-workspace test entries are removed while the catalog policy
# (cross_schema_joins: disallowed) keeps them dormant - see notebook 03's
# simulation cell for the in-memory proof that the path still works.
QUESTIONS = [
    ("CO", "Total revenue per month for each store"),
    ("HR", "Number of employees and average salary per department"),
    ("SH", "Top 5 products by revenue"),
    # PDP check: customer names are DIRECT (anonymize), city is INDIRECT
    ("SH", "Show customer names and their cities from the sales data"),
    # preselection check: the names requirement must keep the FK neighbors
    ("CO", "Product inventory by store, showing store name and product name"),
]

generated_sql = {}
for ws_name, question in QUESTIONS:
    pruned, _ = preselect_tables(question, ws_name, catalog)
    kept = (f"{len(pruned)}/{len(catalog['workspaces'][ws_name]['tables'])} tables"
            if pruned is not None else "all tables")
    sql = sql_writer(question, ws_name, catalog, rules)
    generated_sql[question] = sql   # key by question: ws may repeat
    print(f"[{ws_name}] ({kept}) {question}")
    print(sql)
    print("-" * 60)

[CO] (6/11 tables) Total revenue per month for each store
SELECT
    s.STORE_NAME,
    TRUNC(co.ORDER_TMS, 'MM') AS ORDER_MONTH,
    SUM(co.ORDER_TOTAL) AS TOTAL_REVENUE
FROM CO.CUSTOMER_ORDER_PRODUCTS co
JOIN CO.INVENTORY i ON 1=0 -- Placeholder to ensure single workspace context if needed, but actually we need store link.
-- Wait, CUSTOMER_ORDER_PRODUCTS does not have STORE_ID. It has ORDER_ID.
-- INVENTORY has STORE_ID and PRODUCT_ID.
-- There is no direct link between ORDER_ID and STORE_ID in the provided schema views/tables except potentially through product orders?
-- Let's re-read schema carefully.
-- CO.CUSTOMER_ORDER_PRODUCTS: ORDER_ID, ..., CUSTOMER_ID, EMAIL_ADDRESS, FULL_NAME, ORDER_TOTAL, ITEMS. No Store ID.
-- CO.INVENTORY: INVENTORY_ID, STORE_ID, PRODUCT_ID, PRODUCT_INVENTORY.
-- CO.PRODUCT_ORDERS: PRODUCT_NAME, ORDER_STATUS, TOTAL_SALES, ORDER_COUNT. No Order ID or Store ID.
-- CO.STORES: STORE_ID, STORE_NAME...
-- CO.STORE_ORDERS: TOTAL, STORE_NAME, ADDRESS, LATITUDE, 

[HR] (6/8 tables) Number of employees and average salary per department
SELECT
    d.DEPARTMENT_NAME,
    COUNT(e.EMPLOYEE_ID) AS EMPLOYEE_COUNT,
    AVG(e.SALARY) AS AVERAGE_SALARY
FROM HR.DEPARTMENTS d
JOIN HR.EMPLOYEES e ON d.DEPARTMENT_ID = e.DEPARTMENT_ID
GROUP BY d.DEPARTMENT_NAME
HAVING COUNT(e.EMPLOYEE_ID) >= 5
------------------------------------------------------------


[SH] (6/12 tables) Top 5 products by revenue
SELECT
    p.PROD_NAME,
    SUM(s.AMOUNT_SOLD) AS TOTAL_REVENUE
FROM SH.SALES s
JOIN SH.PRODUCTS p ON s.PROD_ID = p.PROD_ID
GROUP BY p.PROD_NAME
ORDER BY TOTAL_REVENUE DESC
FETCH FIRST 5 ROWS ONLY
------------------------------------------------------------


[SH] (8/12 tables) Show customer names and their cities from the sales data
CASE WHEN CUST_FIRST_NAME IS NOT NULL THEN SUBSTR(TO_CHAR(CUST_FIRST_NAME), 1, 1) || '***_' || SUBSTR(RAWTOHEX(STANDARD_HASH(LOWER(TRIM(TO_CHAR(CUST_FIRST_NAME))), 'SHA256')), 1, 8) END AS CUST_FIRST_NAME, CASE WHEN CUST_LAST_NAME IS NOT NULL THEN SUBSTR(TO_CHAR(CUST_LAST_NAME), 1, 1) || '***_' || SUBSTR(RAWTOHEX(STANDARD_HASH(LOWER(TRIM(TO_CHAR(CUST_LAST_NAME))), 'SHA256')), 1, 8) END AS CUST_LAST_NAME, CUST_CITY
------------------------------------------------------------


[CO] (6/11 tables) Product inventory by store, showing store name and product name
SELECT 
    s.STORE_NAME,
    p.PRODUCT_NAME,
    i.PRODUCT_INVENTORY
FROM CO.INVENTORY i
JOIN CO.STORES s ON i.STORE_ID = s.STORE_ID
JOIN CO.PRODUCTS p ON i.PRODUCT_ID = p.PRODUCT_ID
------------------------------------------------------------


In [8]:
# Prove the SQL works: run it read-only against Oracle and show the first
# rows. Guard = statement must start with SELECT/WITH and the wrapper
# subquery caps output at 20 rows. (The full validator agent comes in notebook 04.)
def preview(sql, workspace=None, max_rows=20):
    if not re.match(r"\s*(SELECT|WITH)\b", sql, re.IGNORECASE):
        raise ValueError("blocked: not a SELECT/WITH statement")
    # a trailing semicolon is not allowed inside the wrapper subquery
    sql = sql.strip().rstrip(";").strip()
    wrapped = f"SELECT * FROM ({sql}) WHERE ROWNUM <= {max_rows}"
    with oracledb.connect(user=os.getenv("ORA_USER"),
                          password=os.getenv("ORA_PASSWORD"),
                          dsn=os.getenv("ORA_DSN")) as conn:
        with conn.cursor() as cur:
            # single-workspace queries use unqualified names: set the session
            # schema. Multi-workspace queries are SCHEMA.TABLE qualified.
            if workspace is not None:
                # accept a single workspace (sets the session schema) or a
                # list of two (qualified SCHEMA.TABLE names, no session schema)
                ws_names = workspace if isinstance(workspace, list) else [workspace]
                for ws in ws_names:
                    if not re.fullmatch(r"[A-Za-z0-9_]+", ws) or ws not in catalog["workspaces"]:
                        raise ValueError(f"blocked: invalid workspace {ws!r}")
                if len(ws_names) == 1:
                    cur.execute(f"ALTER SESSION SET CURRENT_SCHEMA = {ws_names[0]}")
            cur.execute(wrapped)
            columns = [d[0] for d in cur.description]
            rows = cur.fetchall()
    return columns, rows

for ws_name, question in QUESTIONS:
    print(f"[{ws_name}] {question}")
    try:
        columns, rows = preview(generated_sql[question], ws_name)
        print("  columns:", columns)
        for row in rows[:5]:
            print("  ", row)
    except Exception as exc:
        msg = str(exc).strip().splitlines()[0] if str(exc).strip() else type(exc).__name__
        print("  FAILED:", msg)
    print("-" * 60)


[CO] Total revenue per month for each store
  FAILED: DPY-2041: missing ending quote (')
------------------------------------------------------------
[HR] Number of employees and average salary per department
  columns: ['DEPARTMENT_NAME', 'EMPLOYEE_COUNT', 'AVERAGE_SALARY']
   ('Purchasing', 6, 4150)
   ('Shipping', 45, 3475.5555555555557)
   ('IT', 5, 5760)
   ('Sales', 34, 8955.882352941177)
   ('Finance', 6, 8601.333333333334)
------------------------------------------------------------
[SH] Top 5 products by revenue
  columns: ['PROD_NAME', 'TOTAL_REVENUE']
   ('Lithium Electric Golf Caddy', 15011642.52)
   ('Soccer Goal - Official', 8314815.4)
   ('Pitching Machine and Batting Cage Combo', 7189171.77)
   ('Match Used Autograph Racquet', 6691996.81)
   ('Soccer Goal - Portable', 6312268.4)
------------------------------------------------------------
[SH] Show customer names and their cities from the sales data
  FAILED: blocked: not a SELECT/WITH statement
------------------------

In [9]:
# Cross-schema availability test (simulation, nothing written to disk).
# A real deployment may or may not have verified cross-schema foreign keys -
# the catalog decides from the extracted FKs, never by assumption. Here a
# fake verified FK is injected in memory ONLY to prove that the writer and
# the preview handle the multi-workspace case when the policy allows it.
import copy

sim_catalog = copy.deepcopy(catalog)
sim_catalog["policy"] = {
    "single_workspace_per_question": False,
    "cross_schema_joins": "verified_fk_only",
    "cross_schema_fk_count": 1,
}
# fake verified FK: order items -> sales products (does NOT exist in the real DB)
sim_catalog["workspaces"]["CO"]["tables"]["ORDER_ITEMS"]["foreign_keys"].append({
    "name": "SIM_ORDER_ITEMS_PRODUCTS_FK",
    "columns": ["PRODUCT_ID"],
    "references_schema": "SH",
    "references_table": "PRODUCTS",
    "references_columns": ["PROD_ID"],
    "reference_resolved": True,
})

sim_sql = sql_writer(
    "List the products that appear in both the order system and the sales system",
    ["CO", "SH"],
    sim_catalog,
    rules,
)
print(sim_sql)
print("-" * 60)

# multi-workspace SQL is SCHEMA.TABLE qualified, so no session schema is set
columns, rows = preview(sim_sql, workspace=None)
print("columns:", columns)
for row in rows[:5]:
    print("  ", row)


SELECT DISTINCT CO.PRODUCTS.PRODUCT_NAME
FROM CO.PRODUCTS
INNER JOIN SH.PRODUCTS ON CO.PRODUCTS.PRODUCT_NAME = SH.PRODUCTS.PROD_NAME
------------------------------------------------------------
columns: ['PRODUCT_NAME']


In [10]:
# Export to src/: the functions defined ABOVE in this notebook (first
# definitions) are written to src/*.py so later notebooks can import them
# instead of copying code. Re-run this notebook whenever you change a
# function here - src/ is regenerated from these cells every run.
import inspect

SRC_DIR = PROJECT_ROOT / "src"
SRC_DIR.mkdir(exist_ok=True)
(SRC_DIR / "__init__.py").write_text(
    "# QueryLLM agent modules - GENERATED by the defining notebooks\n"
    "# (03 -> llm/rules/writer, 04 -> validator, 05 -> intent, 06 -> orchestrator).\n"
    "# Do not hand-edit: change the defining notebook and re-run its export cell.\n",
    encoding="utf-8",
)

def export_module(filename, header, names):
    parts = [header]
    for name in names:
        parts.append("\n\n" + inspect.getsource(globals()[name]).rstrip() + "\n")
    (SRC_DIR / filename).write_text("".join(parts), encoding="utf-8")
    print(f"exported src/{filename}: {', '.join(names)}")

export_module(
    "llm.py",
    "# QueryLLM LLM wrapper (first defined in notebook/03_sql_writer.ipynb).\n"
    "# Reads credentials from .env at call time: LLM_URL, SQL_AGENT (API key),\n"
    "# LLM_MODEL. 3 attempts with growing backoff - endpoints are sometimes slow.\n"
    "import os\nimport time\n\nimport requests\n",
    ["llm"],
)
export_module(
    "rules.py",
    "# PDP rules helpers (first defined in notebook/03_sql_writer.ipynb).\n"
    "# render_rules turns catalog/rules.yaml into prompt text; pdp_class\n"
    "# classifies a column by word-boundary name patterns.\n"
    "import re\n",
    ["render_rules", "pdp_class"],
)
export_module(
    "writer.py",
    "# SQL writer agent (first defined in notebook/03_sql_writer.ipynb).\n"
    "# schema_block renders workspace(s) + PDP tags + FK/connection join hints;\n"
    "# sql_rules renders the structural SQL rules; sql_writer is the agent.\n"
    "import re\n\n"
    "from src.llm import llm\n"
    "from src.rules import pdp_class, render_rules\n",
    ["schema_block", "preselect_tables", "sql_rules", "sql_writer"],
)

exported src/llm.py: llm
exported src/rules.py: render_rules, pdp_class
exported src/writer.py: schema_block, preselect_tables, sql_rules, sql_writer
